# cv_r1 配布バンドル展開・配置・検証【Colab】

`cadence_cv_r1_meta_v*.tgz`（必須）＋`cadence_cv_r1_wavs_v*.tar`（wav。既に Drive にあれば省略可）を
**学習ルート**（train 実行 cwd。`Data/cv_r1` がその直下になる場所）へ展開し、整合を検証する。

入手経路は3通り（§1 で指定）:
1. **URL**（公開リリース: Zenodo / Hugging Face / GitHub Release 等）→ ダウンロードして展開
2. **Drive 上のアーカイブ**（事前に Drive へアップロード済み）→ そのまま展開
3. **Colab へ直接アップロード**（`/content` に置く。ランタイム消滅で消えるので小さい meta 向き）

使い方の対応:
- **(a) 全部自作**: 公開パイプライン（手順1〜6）で `Data/cv_r1` を再生成 → 本ノートは §3 検証だけ使う
- **(b) そのまま使う**: meta + wavs を展開 → §3 検証 → 学習
- **(c) 同形式で自作**: `README_bundle.md` の形式仕様どおりに作成 → §3 検証だけ使う
- **(d) 部分差し替え**: meta のみ展開し直し（wav は既存流用）等


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

# ===== §1 設定 =====
# 学習ルート = Drive 上の fork clone（layer-b-cadence-seq）直下。train_ms_jp_extra の実行 cwd で、この直下に Data/cv_r1 が置かれる
BASE = Path('/content/drive/MyDrive/Style-Bert-VITS2')   # ←要確認・要修正
BASE.mkdir(parents=True, exist_ok=True)

# アーカイブの所在: URL / Drive・Colab 上のパス のどちらかを指定（None は「使わない」）
# 公開配布は Zenodo（CC0・DOI 付き）。直リンク形式:
#   https://zenodo.org/records/<recid>/files/<ファイル名>?download=1
#   <recid> はレコード公開後の URL 末尾の数字。ファイル名は完全一致（大文字小文字も）。
META_SRC = 'https://zenodo.org/records/21119791/files/cadence_cv_r1_meta_v20260702.tgz?download=1'   # DOI 10.5281/zenodo.21119791
WAVS_SRC = 'https://zenodo.org/records/21119791/files/cadence_cv_r1_wavs_v20260702.tar?download=1'   # wav が既に配置済みなら None

print('BASE:', BASE)


In [ ]:
# ===== §2 取得・展開 =====
import tarfile, urllib.request, os, sys

def fetch(src):
    if src is None: return None
    s = str(src)
    if s.startswith('http://') or s.startswith('https://'):
        from urllib.parse import urlparse
        dst = Path('/content') / Path(urlparse(s).path).name   # クエリ(?download=1)を除いた名前
        if not dst.exists():
            print('download:', s)
            urllib.request.urlretrieve(s, dst)
        return dst
    p = Path(s)
    assert p.exists(), f'見つからない: {p}'
    return p

def extract(arch):
    if arch is None: return
    print('extract:', arch.name, '->', BASE)
    with tarfile.open(arch) as tf:
        names = tf.getnames()
        assert all(n.startswith('Data/cv_r1') for n in names), f'想定外のアーカイブ構造: {names[:3]}'
        tf.extractall(BASE)
    print(f'  {len(names)} entries')

extract(fetch(META_SRC))
extract(fetch(WAVS_SRC))
DATA = BASE / 'Data' / 'cv_r1'
assert DATA.is_dir(), f'展開失敗: {DATA} が無い'
print('OK:', DATA)


## §3 検証（配置整合＝学習開始ゲート）
1. MANIFEST.sha256（meta 系ファイル＋cadseq 全件）
2. config.json spk2id が sorted 採番と一致・n_speakers=298
3. esd 14226/3789・7列・話者 298
4. wav 18015 実在（esd 相対パスで解決）
5. cadseq: train present=11010 / missing=3216（zeros fallback）/ shape==(P,32)、val present=0
6. 非ゼロ行 ‖row‖≈1・wav サンプリングレート確認


In [ ]:
import json, hashlib, collections, random, wave
import numpy as np

# 1) MANIFEST
def sha256(p, bufsize=1<<20):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for c in iter(lambda: f.read(bufsize), b''): h.update(c)
    return h.hexdigest()

bad = 0; n_checked = 0
for line in open(DATA/'MANIFEST.sha256', encoding='utf-8'):
    line = line.strip()
    if not line or line.startswith('#'): continue
    h, name = line.split(None, 1)
    p = BASE / name
    if not p.exists(): print('  missing:', name); bad += 1; continue
    if sha256(p) != h: print('  hash NG:', name); bad += 1
    n_checked += 1
print(f'MANIFEST: {n_checked} 件検査 / NG {bad}（0 が正）'); assert bad == 0

# 2) config / spk2id
cfg = json.load(open(DATA/'config.json', encoding='utf-8'))
def read_esd(p):
    n=0; spks=set(); bad7=0
    for l in open(p, encoding='utf-8'):
        l=l.rstrip('\n')
        if not l: continue
        f=l.split('|'); n+=1; spks.add(f[1])
        if len(f)!=7: bad7+=1
    return n, spks, bad7
ntr, s_tr, b1 = read_esd(DATA/'esd_train.list')
nvl, s_vl, b2 = read_esd(DATA/'esd_val.list')
spks = sorted(s_tr | s_vl); ref = {s:i for i,s in enumerate(spks)}
print(f'esd: train {ntr}(期待14226) / val {nvl}(期待3789) / 話者 {len(spks)}(期待298) / 7列違反 {b1+b2}(0 が正)')
print(f"spk2id sorted一致: {cfg['data']['spk2id']==ref} / n_speakers={cfg['data'].get('n_speakers')}")
assert b1==b2==0 and ntr==14226 and nvl==3789 and len(spks)==298 and cfg['data']['spk2id']==ref

# 3-5) wav / cadseq
def check_list(p, name):
    miss_wav=0; present=0; missing=0; mism=0; okp=[]
    for l in open(p, encoding='utf-8'):
        l=l.rstrip('\n')
        if not l: continue
        f=l.split('|'); wav=f[0]; nph=len(f[4].split())
        w = Path(wav) if wav.startswith('/') else BASE/wav
        if not w.exists(): miss_wav+=1
        npy = Path(str(w)+'.cadseq.npy')
        if npy.exists():
            if np.load(npy, mmap_mode='r').shape==(nph,32): present+=1; okp.append(npy)
            else: mism+=1
        else: missing+=1
    print(f'{name}: wav欠損={miss_wav}(0 が正) cadseq present={present} missing={missing} shape_mismatch={mism}(0 が正)')
    return okp, miss_wav, mism

ok1, mw1, ms1 = check_list(DATA/'esd_train.list', 'esd_train')   # present 期待 11010
ok2, mw2, ms2 = check_list(DATA/'esd_val.list',   'esd_val')     # present 期待 0
assert mw1==mw2==ms1==ms2==0 and len(ok1)==11010 and len(ok2)==0

# 6) 値・sr サンプル
for npy in random.sample(ok1, 3):
    a=np.load(npy); nz=a.any(axis=1)
    print(f'  {npy.name}: shape={a.shape} ‖row‖mean={np.linalg.norm(a[nz],axis=1).mean():.3f}')
some_wav = BASE / next(l for l in open(DATA/'esd_train.list', encoding='utf-8') if l.strip()).split('|')[0]
with wave.open(str(some_wav),'rb') as w:
    print(f'  wav sr={w.getframerate()} ch={w.getnchannels()}')

print('\n=== 全チェック通過: 学習開始可（branch layer-b-cadence-seq）===')


## 失敗時
- `MANIFEST NG` → アーカイブ再取得（転送破損）。
- `spk2id 不一致`／`shape_mismatch`／`wav欠損` → **学習開始禁止**。配置をやり直す。
- 自作データ（使い方 c）は MANIFEST が無いので §3 の 1) をスキップし 2) 以降で検証。

## 学習前の残工程（Colab / fork 側）
- `preprocess_text` は不要（esd_train/val をバンドルで配置済み。実行すると spk2id を再生成し得るので走らせない）。
- **`bert_gen` と `style_gen` は必要**（.bert.pt / style ベクトルは重量のためバンドル非同梱）。R1 と同手順で実行してから `train_ms_jp_extra`。
